In [1]:
print("hello")

hello


In [2]:
import re
import pdfplumber
import pandas as pd

PDF = "../dataraw/All_India_Electricity_Statistics.pdf"

with pdfplumber.open(PDF) as pdf:
    text = pdf.pages[31].extract_text()   # Python counts from 0, so page 32 = index 31

print(text)

Central Electricity Authority General Review 2025
Table 1.7
Growth of Total Electricity Consumption (GWh) by Ultimate Consumers
State-Wise
Utilities Only
State/UTs 2012-13 2013-14 2014-15 2015-16 2016-17 2017-18 2018-19 2019-20 2020-21 2021-22 2022-23 2023-24
Chandigarh 1365 1419 1472 1497 1589 1574 1535 1604 1349 1409 1646 1589
Delhi 23144 23981 24775 25658 26604 28488 29169 29812 26319 27625 34107 32300
Haryana 26258 29083 32197 35092 34501 38215 41017 43382 41944 46231 55241 55734
Himachal Pradesh 7358 7649 7842 7958 7974 8405 9041 9124 8635 10198 10976 11354
Jammu & Kashmir 5163 5754 6372 7144 7902 8395 9633 10623 9984 10231 9769 10610
Punjab 35825 37557 39384 40843 44050 47027 50120 49168 50286 53489 58532 59712
Rajasthan 42160 43151 49920 50359 51027 53618 60151 61942 64888 69831 79615 83789
Uttar Pradesh 55550 59177 64707 70967 81068 90395 91355 94932 93600 98730 115675 121122
Uttarakhand 8574 9597 9980 10298 11162 11210 11543 12021 11225 12519 13509 13875
Sub-Total (NR) 205397 

In [3]:
YEARS = ["2012-13","2013-14","2014-15","2015-16","2016-17","2017-18",
         "2018-19","2019-20","2020-21","2021-22","2022-23","2023-24"]
MERGED_UT = "Dadra & Nagar Haveli and Daman & Diu"

# one standard spelling per state, used in EVERY table (so joins work later)
NAME_FIX = {
    "Madhay Pradesh": "Madhya Pradesh", "Orissa": "Odisha",
    "A & N Islands": "Andaman & Nicobar Islands",
    "Chhatisgarh": "Chhattisgarh", "Pondicherry": "Puducherry",
    "Lakshwadeep": "Lakshadweep",
}

In [4]:
NUM = re.compile(r"^\d[\d,]*\.?\d*$")   # looks like 1365 or 4508.32 or 1,365

def split_line(line):
    tokens = line.split()
    i = len(tokens)
    while i > 0 and NUM.match(tokens[i-1]):   # walk left while tokens are numbers
        i -= 1
    name = " ".join(tokens[:i])
    nums = [float(t.replace(",", "")) for t in tokens[i:]]
    return name, nums

print(split_line("Delhi 23144 23981 24775"))

('Delhi', [23144.0, 23981.0, 24775.0])


In [5]:
rows, leftovers = {}, []
for line in text.split("\n"):
    name, nums = split_line(line)
    if len(nums) < 2:
        continue                      # title and header lines
    if not name:
        leftovers.append(nums)        # number-only lines (special case)
    else:
        rows[name] = nums

print(len(rows), "named rows;", len(leftovers), "number-only lines")
for k, v in rows.items():
    if len(v) != 12:
        print("irregular:", k, len(v))

41 named rows; 3 number-only lines
irregular: Daman & Diu 2
irregular: Andhra Pradesh 10
irregular: Telangana 10


In [6]:
old_a, old_b, ap_early = leftovers
merged_early = [a + b for a, b in zip(old_a, old_b)]   # add the two UTs, year by year

# swap the Daman & Diu row IN PLACE (keeps the row order, which we need for regions)
rows = {(MERGED_UT if k == "Daman & Diu" else k):
        (merged_early + v if k == "Daman & Diu" else v)
        for k, v in rows.items()}

rows["Andhra Pradesh"] = ap_early + rows["Andhra Pradesh"]   # 2 + 10 = 12 values
rows["Telangana"] = [None, None] + rows["Telangana"]

In [7]:
REGIONS = {"NR": "Northern", "WR": "Western", "SR": "Southern",
           "ER": "Eastern", "NER": "North-Eastern"}
records, bucket = [], []

for name in rows:
    if name.startswith("Sub-Total"):
        code = re.search(r"\((\w+)\)", name).group(1)
        for j, yr in enumerate(YEARS):                       # validation
            s = sum(rows[n][j] for n in bucket if rows[n][j] is not None)
            if abs(s - rows[name][j]) > 2:
                print(f"CHECK {code} {yr}: sum {s:.0f} vs printed {rows[name][j]:.0f}")
        for n in bucket:
            for j, yr in enumerate(YEARS):
                records.append((n, REGIONS[code], yr, rows[n][j]))
        bucket = []
    elif not name.startswith("Total"):
        bucket.append(name)

cons = pd.DataFrame(records, columns=["state", "region", "fiscal_year", "consumption_gwh"])
cons["state"] = cons["state"].replace(NAME_FIX)

print(cons.shape, cons["state"].nunique(), "states/UTs")
print(cons.head())

(420, 4) 35 states/UTs
        state    region fiscal_year  consumption_gwh
0  Chandigarh  Northern     2012-13           1365.0
1  Chandigarh  Northern     2013-14           1419.0
2  Chandigarh  Northern     2014-15           1472.0
3  Chandigarh  Northern     2015-16           1497.0
4  Chandigarh  Northern     2016-17           1589.0


In [8]:
cons.groupby("fiscal_year")["consumption_gwh"].sum()   # 2023-24 should be about 1,350,092

fiscal_year
2012-13     708843.00
2013-14     751911.00
2014-15     814253.00
2015-16     863364.00
2016-17     914091.00
2017-18     973134.00
2018-19    1037517.00
2019-20    1052347.00
2020-21    1041653.00
2021-22    1141484.00
2022-23    1264103.32
2023-24    1350094.00
Name: consumption_gwh, dtype: float64

In [10]:
cons.to_csv("../dataclean/consumption_clean.csv", index=False)

In [11]:
with pdfplumber.open(PDF) as pdf:
    tbl = pdf.pages[65].extract_tables()[0]          # page 66 = index 65

data = [r for r in tbl[3:] if r[1] and not r[1].startswith("Total")]   # drop 3 header rows and the total row
cols = ["sno", "state", "small_hydro_mw", "wind_mw", "bm_bagasse_mw", "bm_non_bagasse_mw",
        "waste_to_energy_mw", "waste_to_energy_offgrid_mw", "bio_total_mw", "solar_mw", "total_re_mw"]
re_df = pd.DataFrame(data, columns=cols).drop(columns="sno")

for c in re_df.columns[1:]:
    re_df[c] = pd.to_numeric(re_df[c], errors="coerce").fillna(0.0)   # blank cell = 0 MW

re_df["state"] = re_df["state"].str.strip().replace(NAME_FIX)
re_df["state"] = re_df["state"].replace({"Dadar & Nagar Haveli": "Dadra & Nagar Haveli"})

print(re_df.shape)
print(re_df[["small_hydro_mw", "wind_mw", "bio_total_mw", "solar_mw", "total_re_mw"]].sum().round(2))

(38, 10)
small_hydro_mw      5003.25
wind_mw            45886.51
bio_total_mw       10941.15
solar_mw           81813.60
total_re_mw       143644.51
dtype: float64


In [12]:
re_df["state"] = re_df["state"].replace({"Dadra & Nagar Haveli": MERGED_UT,
                                         "Daman & Diu": MERGED_UT,
                                         "Ladakh": "Jammu & Kashmir"})
re_df = re_df[re_df["state"] != "Others"]
num_cols = [c for c in re_df.columns if c != "state"]
re_df = re_df.groupby("state", as_index=False)[num_cols].sum()

print("in RE but not in consumption:", sorted(set(re_df.state) - set(cons.state)))
print("in consumption but not in RE:", sorted(set(cons.state) - set(re_df.state)))
print(len(re_df), "rows")

re_df.to_csv("../dataclean/re_capacity_clean.csv", index=False)

in RE but not in consumption: ['Andaman & Nicobar']
in consumption but not in RE: ['Andaman & Nicobar Islands']
35 rows


In [13]:
re_df["state"] = re_df["state"].replace({"Andaman & Nicobar": "Andaman & Nicobar Islands"})
re_df = re_df.groupby("state", as_index=False)[num_cols].sum()

print("in RE but not in consumption:", sorted(set(re_df.state) - set(cons.state)))
print("in consumption but not in RE:", sorted(set(cons.state) - set(re_df.state)))
print(len(re_df), "rows")

re_df.to_csv("../dataclean/re_capacity_clean.csv", index=False)

in RE but not in consumption: []
in consumption but not in RE: []
35 rows


In [14]:
def split_line_clean(line):
    name, nums = split_line(line)               # helper from Section 3
    return name.replace(" $", "").strip(), nums   # strip the DVC footnote mark

def read_sales_page(page_no):
    with pdfplumber.open(PDF) as pdf:
        text = pdf.pages[page_no - 1].extract_text()
    out = {}
    for line in text.split("\n"):
        name, nums = split_line_clean(line)
        if name and len(nums) == 5:             # every data row has exactly 5 numbers
            out[name] = nums
    return out

p1, p2 = read_sales_page(139), read_sales_page(140)
print(len(p1), len(p2), set(p1) == set(p2))

41 41 True


In [15]:
cols1 = ["domestic", "commercial", "industrial_lv_mv", "industrial_hv", "public_lighting"]
cols2 = ["traction", "agriculture", "water_works", "miscellaneous", "total_sold"]
df = pd.DataFrame({k: p1[k] + p2[k] for k in p1}, index=cols1 + cols2).T
print(df.shape)        # (41, 10)

(41, 10)


In [16]:
states = df[~df.index.str.startswith(("Sub-Total", "Total"))].copy()
sizes = {"NR": 9, "WR": 6, "SR": 7, "ER": 6, "NER": 7}      # states per region, in PDF order
i = 0
for code, n in sizes.items():
    block = states.iloc[i:i + n]; i += n
    gap = (block.sum() - df.loc[f"Sub-Total ({code})"]).abs().max()
    print(code, "largest gap vs printed sub-total:", round(gap, 2))

row_gap = (states[cols1 + cols2[:-1]].sum(axis=1) - states["total_sold"]).abs().max()
print("categories add up to total sold, largest gap:", round(row_gap, 2))

NR largest gap vs printed sub-total: 0.02
WR largest gap vs printed sub-total: 0.01
SR largest gap vs printed sub-total: 0.02
ER largest gap vs printed sub-total: 0.01
NER largest gap vs printed sub-total: 0.02
categories add up to total sold, largest gap: 0.02


In [17]:
NAME_FIX.update({"UT of J&K and Ladakh": "Jammu & Kashmir", "DNH & DD": MERGED_UT})
states.index = [NAME_FIX.get(s, s) for s in states.index]
states.index.name = "state"
states = states.reset_index()

states["ci_gwh"] = states.commercial + states.industrial_lv_mv + states.industrial_hv
states["ci_share_pct"] = states.ci_gwh / states.total_sold * 100

print(states.shape)                                # (35, 13)
print(set(states.state) ^ set(cons.state))         # names that do not match Table 1.7: expect set()
states.to_csv("../dataclean/sales_category_clean.csv", index=False)

(35, 13)
set()
